# Programar Jobs con Prefect desde JupyterHub

Este notebook demuestra cómo usar el **Prefect Server** (ya desplegado en Docker Compose) desde un notebook en JupyterHub para:
- Conectarse a la API de Prefect
- Crear y ejecutar flows (tareas programadas)
- Monitorear ejecuciones
- Acceder a MinIO para datos

**Endpoint Prefect:** `http://prefect:4200/api` (interno) o `http://SERVER_IP:4200/api` (externo)

## 1. Instalar e importar dependencias de Prefect

In [ ]:
import subprocess
import sys

# Verificar que Prefect esté instalado, si no, instalarlo
try:
    import prefect
    print(f"✓ Prefect {prefect.__version__} ya está instalado")
except ImportError:
    print("Instalando Prefect...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "prefect"])
    import prefect
    print(f"✓ Prefect {prefect.__version__} instalado exitosamente")

# Importar librerías necesarias
from prefect import flow, task
from prefect.client.orchestration import get_client
from prefect.client.schemas.objects import State
import httpx
import os
from datetime import datetime, timedelta
import json

print("✓ Todas las dependencias importadas exitosamente")

## 2. Configurar conexión desde JupyterHub al Prefect Server

Definir la URL del Prefect API y variables de entorno necesarias.

In [ ]:
# Configuración del Prefect API
# Dentro de Docker Compose, el servicio Prefect es accesible en http://prefect:4200/api
# Si necesitas acceder desde fuera, usa http://SERVER_IP:4200/api

PREFECT_API_URL = os.getenv("PREFECT_API_URL", "http://prefect:4200/api")
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minioadmin")

print(f"📍 PREFECT_API_URL: {PREFECT_API_URL}")
print(f"📍 MINIO_ENDPOINT: {MINIO_ENDPOINT}")

# Configurar Prefect para usar el servidor
os.environ["PREFECT_API_URL"] = PREFECT_API_URL

## 3. Validar conectividad y versión de API

In [ ]:
import asyncio

async def test_connection():
    """Validar conexión al Prefect Server"""
    try:
        # Obtener el cliente de Prefect
        client = await get_client(api_url=PREFECT_API_URL)
        
        # Intentar obtener la versión del servidor
        health_response = httpx.get(f"{PREFECT_API_URL}/health")
        if health_response.status_code == 200:
            print("✓ Conexión exitosa a Prefect Server")
            print(f"  Status: {health_response.json()}")
        
        return True
    except Exception as e:
        print(f"✗ Error conectando a Prefect: {e}")
        return False

# Ejecutar validación
success = asyncio.run(test_connection())
if not success:
    print("⚠️  Verifica que PREFECT_API_URL esté correcta y que el servicio esté ejecutándose")

## 4. Consultar work pools, work queues y deployments existentes

In [ ]:
async def list_deployments():
    """Listar todos los deployments disponibles en el servidor Prefect"""
    try:
        client = await get_client(api_url=PREFECT_API_URL)
        
        # Listar deployments
        deployments = await client.read_deployments()
        
        print(f"📋 Total de deployments: {len(deployments)}")
        for dep in deployments:
            print(f"   - {dep.name} (ID: {dep.id})")
            if dep.description:
                print(f"     Descripción: {dep.description}")
        
        return deployments
    except Exception as e:
        print(f"✗ Error listando deployments: {e}")
        return []

async def list_work_pools():
    """Listar todos los work pools disponibles"""
    try:
        client = await get_client(api_url=PREFECT_API_URL)
        work_pools = await client.read_work_pools()
        
        print(f"\n🏊 Total de work pools: {len(work_pools)}")
        for pool in work_pools:
            print(f"   - {pool.name} (Tipo: {pool.type})")
        
        return work_pools
    except Exception as e:
        print(f"✗ Error listando work pools: {e}")
        return []

# Ejecutar consultas
deployments = asyncio.run(list_deployments())
work_pools = asyncio.run(list_work_pools())

## 5. Ejecutar un deployment existente desde el notebook

Si ya tienes un deployment guardado en Prefect, puedes dispararlo desde aquí.

In [ ]:
async def trigger_deployment(deployment_name: str, params: dict = None):
    """
    Ejecutar un deployment existente
    
    Args:
        deployment_name: Nombre del deployment (ej: "my-flow/my-deployment")
        params: Parámetros a pasar al flow (opcional)
    """
    try:
        client = await get_client(api_url=PREFECT_API_URL)
        
        # Buscar el deployment por nombre
        deployments = await client.read_deployments(
            deployment_filter={"name": {"like_": deployment_name}}
        )
        
        if not deployments:
            print(f"✗ No se encontró deployment con nombre: {deployment_name}")
            return None
        
        deployment = deployments[0]
        print(f"📤 Ejecutando deployment: {deployment.name}")
        
        # Crear una flow run
        flow_run = await client.create_flow_run_from_deployment(
            deployment.id,
            parameters=params or {}
        )
        
        print(f"✓ Flow run creada con ID: {flow_run.id}")
        print(f"  Estado: {flow_run.state.type if flow_run.state else 'No state'}")
        
        return flow_run.id
        
    except Exception as e:
        print(f"✗ Error ejecutando deployment: {e}")
        return None

# Ejemplo de uso (descomenta si tienes un deployment):
# flow_run_id = asyncio.run(trigger_deployment("my-flow/my-deployment", {"param1": "value1"}))

## 6. Definir y ejecutar un flow local apuntando al servidor

In [ ]:
# Definir tareas (tasks)
@task
def load_data_from_minio(bucket: str, key: str):
    """Cargar datos desde MinIO"""
    print(f"📂 Cargando {key} desde bucket {bucket}...")
    # Aquí iría lógica real para acceder a MinIO
    return {"bucket": bucket, "key": key, "status": "loaded"}

@task
def process_data(data: dict):
    """Procesar datos"""
    print(f"⚙️  Procesando datos...")
    processed = {**data, "processed_at": datetime.now().isoformat()}
    return processed

@task
def save_results(processed_data: dict):
    """Guardar resultados"""
    print(f"💾 Guardando resultados...")
    print(f"   Datos: {processed_data}")
    return {"status": "saved"}

# Definir el flow
@flow(name="data-pipeline-job")
def data_pipeline(bucket: str = "raw-data", key: str = "sample.csv"):
    """Flow principal que orquesta el pipeline de datos"""
    data = load_data_from_minio(bucket, key)
    processed = process_data(data)
    result = save_results(processed)
    return result

# Ejecutar el flow
print("🚀 Ejecutando flow de ejemplo...")
result = data_pipeline()
print(f"✓ Flow completado: {result}")

## 7. Monitorear estados de ejecución y recuperar logs

In [ ]:
async def monitor_flow_run(flow_run_id: str, max_wait_seconds: int = 60):
    """
    Monitorear el estado de una flow run hasta que termine
    
    Args:
        flow_run_id: ID de la flow run a monitorear
        max_wait_seconds: Tiempo máximo a esperar (segundos)
    """
    try:
        client = await get_client(api_url=PREFECT_API_URL)
        
        start_time = datetime.now()
        terminal_states = {"COMPLETED", "FAILED", "CANCELLED"}
        
        print(f"⏱️  Monitoreando flow run: {flow_run_id}")
        
        while True:
            # Obtener estado actual
            flow_run = await client.read_flow_run(flow_run_id)
            state = flow_run.state
            
            print(f"  Estado: {state.type} | Timestamp: {datetime.now().strftime('%H:%M:%S')}")
            
            # Si está en estado terminal, salir
            if state.type in terminal_states:
                print(f"✓ Flow run finalizada con estado: {state.type}")
                if state.message:
                    print(f"  Mensaje: {state.message}")
                return flow_run
            
            # Revisar timeout
            elapsed = (datetime.now() - start_time).total_seconds()
            if elapsed > max_wait_seconds:
                print(f"⏱️  Timeout después de {elapsed:.0f} segundos")
                return flow_run
            
            # Esperar antes de siguiente check
            await asyncio.sleep(5)
            
    except Exception as e:
        print(f"✗ Error monitoreando flow run: {e}")
        return None

async def get_flow_run_logs(flow_run_id: str):
    """Obtener logs de una flow run"""
    try:
        client = await get_client(api_url=PREFECT_API_URL)
        
        # Intentar obtener logs
        logs = await client.read_logs(
            filter={"flow_run_id": {"eq_": flow_run_id}}
        )
        
        print(f"\n📝 Logs de flow run {flow_run_id}:")
        for log in logs:
            print(f"  [{log.timestamp}] {log.level}: {log.message}")
        
        return logs
    except Exception as e:
        print(f"✗ Error obteniendo logs: {e}")
        return []

# Ejemplo: monitorear un flow run (si tienes el ID)
# flow_run = asyncio.run(monitor_flow_run("your-flow-run-id"))
# logs = asyncio.run(get_flow_run_logs("your-flow-run-id"))

## 8. Manejo básico de errores de conexión y reintentos

In [ ]:
async def safe_api_call(func, max_retries: int = 3, retry_delay: int = 5):
    """
    Ejecutar una función con reintentos automáticos
    
    Args:
        func: Función asíncrona a ejecutar
        max_retries: Número máximo de intentos
        retry_delay: Segundos a esperar entre reintentos
    """
    last_error = None
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Intento {attempt}/{max_retries}...")
            result = await func()
            print(f"✓ Exitoso en intento {attempt}")
            return result
        
        except httpx.ConnectError as e:
            last_error = f"Error de conexión: {e}"
            print(f"✗ {last_error}")
        
        except httpx.TimeoutException as e:
            last_error = f"Timeout: {e}"
            print(f"✗ {last_error}")
        
        except Exception as e:
            last_error = f"Error inesperado: {e}"
            print(f"✗ {last_error}")
        
        # Esperar antes de reintentar
        if attempt < max_retries:
            print(f"  Esperando {retry_delay}s antes del próximo intento...")
            await asyncio.sleep(retry_delay)
    
    print(f"✗ Error después de {max_retries} intentos")
    print(f"  Último error: {last_error}")
    return None

# Ejemplo de uso con reintentos
async def test_with_retries():
    """Test de conexión con reintentos"""
    async def check_health():
        response = httpx.get(f"{PREFECT_API_URL}/health", timeout=10)
        return response.json()
    
    result = await safe_api_call(check_health)
    return result

# test_result = asyncio.run(test_with_retries())

## Próximos pasos

### 📋 Guardar un Flow como Deployment en Prefect Server

Para que un flow se ejecute periódicamente en el servidor, debes guardarlo como **deployment**. Opción A: desde el notebook (si tienes permisos)

```python
import asyncio
from prefect.client.orchestration import get_client

async def save_deployment():
    client = await get_client()
    # Aquí van los detalles del deployment
    # Consulta: https://docs.prefect.io/latest/deploy/index.html

asyncio.run(save_deployment())
```

Opción B: desde terminal (más recomendado)

```bash
# Crear un archivo flows/my_pipeline.py con el @flow
# Luego:
prefect deploy flows/my_pipeline.py:main_flow \
  --name "my-deployment" \
  --pool "default"
```

### ⏰ Crear un Schedule

Una vez desplegado, accede a **Prefect UI** en `http://SERVER_IP:4200` y:
1. Busca tu deployment
2. Haz clic en "Create Schedule"
3. Elige intervalo: cron, interval, etc.

### 🔗 Acceder a MinIO desde dentro de un Flow

```python
import boto3

@task
def read_from_minio(bucket: str, key: str):
    s3 = boto3.client(
        's3',
        endpoint_url='http://minio:9000',
        aws_access_key_id=os.getenv('MINIO_ACCESS_KEY'),
        aws_secret_access_key=os.getenv('MINIO_SECRET_KEY')
    )
    obj = s3.get_object(Bucket=bucket, Key=key)
    return obj['Body'].read()
```

### 📚 Documentación útil
- [Prefect Docs](https://docs.prefect.io)
- [Deployments & Schedules](https://docs.prefect.io/latest/concepts/deployments/)
- [Work Pools](https://docs.prefect.io/latest/concepts/work-pools/)


## ✅ Resumen rápido

| Paso | Descripción | Sección |
|------|-------------|---------|
| 1️⃣ | Instalar Prefect SDK | #1 |
| 2️⃣ | Configurar URL del servidor | #2 |
| 3️⃣ | Validar conexión | #3 |
| 4️⃣ | Listar deployments existentes | #4 |
| 5️⃣ | Ejecutar un deployment | #5 |
| 6️⃣ | Crear un flow nuevo | #6 |
| 7️⃣ | Monitorear ejecuciones | #7 |
| 8️⃣ | Manejo de errores | #8 |

---

**TL;DR:** 
- Prefect Server ya corre en `http://prefect:4200/api`
- Define flows con `@flow` y `@task`
- Ejecuta con `flow_name()` en el notebook
- Guarda como deployment para ejecutar periódicamente
- Accede a UI en `http://SERVER_IP:4200` para schedules